[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/41.%20The%20Terminal%20Concession%20Bidding%20Problem/P41-Tier-3_executed.ipynb)

---


# 41. The Terminal Concession Bidding Problem

## Tier 3: Ant Colony Optimization for Multi-Objective Bidding

### Goal

This notebook applies Ant Colony Optimization (ACO) to a simplified terminal-concession bidding model. Artificial ants build bidding strategies by selecting one option from each of five decision categories. Pheromone trails and heuristic information guide these selections over repeated iterations. A weighted fitness score guides the search, while a separate Pareto analysis identifies non-dominated trade-offs among win-probability score, modeled cost burden, and risk index.

The inputs are the decision options and their illustrative attributes in the decision graph, together with the ACO parameters. The outputs are the best-found strategy, its convergence history, pheromone history, and a Pareto frontier of strategies encountered during the search. The search is stochastic, and no random seed is specified, so repeated executions can produce different search histories and results.

### Scope and modeling assumptions

- A complete strategy selects exactly one option from each of five decision categories. The categories are treated as independent during construction; interaction effects are included only in the win-probability calculation.
- Option contributions to win probability and the synergy bonuses are illustrative model inputs, not calibrated estimates of real-world probabilities.
- `cost` is an additive modeled burden used to compare options. Its input values combine quantities with different native meanings, including monetary amounts and commitments. Therefore, `total_cost` and the associated cost-efficiency score are model indices, not monetary estimates or financial forecasts.
- Risk is an additive index. The model does not include uncertainty distributions, contractual feasibility checks, market responses, or constraints beyond selecting one option from each category.
- The weighted fitness score guides the search; it does not turn the Pareto frontier into a single-objective ranking. A strategy is Pareto-dominated if another strategy is at least as good on every objective and strictly better on at least one.
- The Pareto frontier represents distinct objective vectors observed during the stochastic search. Strategies with the same win-probability score, cost burden, and risk index have the same position in objective space, so only one representative of such a tie is retained.

### Required background: ACO and the construction state

An ant builds a strategy one decision category at a time. Its **state** at any step is the partial strategy it has already selected; its available actions are the options in the next category. In this implementation, selections are made independently within each category, so an option's selection probabilities depend on that category's pheromone and heuristic values, not on the partial strategy.

Pheromone, denoted by $\tau_{j,k}$, is a learned preference for option $k$ in decision category $j$. The heuristic value, denoted by $\eta_{j,k}$, is an option-specific guide calculated from the input attributes. An option with greater pheromone or heuristic value has greater attractiveness. The parameter $q_0$ controls whether the ant chooses the most probable option (exploitation) or samples from the probability distribution (exploration). When multiple options have the same maximum probability, the implementation selects the first one in the category's option list.

After all ants construct strategies in an iteration, pheromone evaporates and strategies deposit pheromone on the options they used. Evaporation reduces the influence of older information; positive-fitness strategies reinforce their selected options. This stochastic metaheuristic uses learned preferences to guide a search; it does not prove that a globally optimal strategy has been found.

### Mathematical model

Let $J$ be the set of decision categories, and let $K_j$ be the available options in category $j$. A complete strategy $s$ selects one option $k_j(s) \in K_j$ for each category $j \in J$. For option $k$ in category $j$, let $c_{j,k}$ be its modeled cost burden, $r_{j,k}$ its risk contribution, and $p_{j,k}$ its win-probability contribution.

The strategy's modeled cost and risk are additive:

$$
C(s) = \sum_{j \in J} c_{j,k_j(s)},
\qquad
R(s) = \sum_{j \in J} r_{j,k_j(s)}.
$$

Here, $C(s)$ is the total modeled cost burden and $R(s)$ is the total risk index. In the code, these quantities are accumulated as `total_cost` and `total_risk` while `construct_solution` selects one option from each category.

The win-probability score combines option contributions with interaction adjustments:

$$
P(s) = \min\!\left(0.95,
0.2\sum_{j \in J} p_{j,k_j(s)} + B(s)\right),
$$

where $B(s)$ is the sum of the specified synergy bonuses when their option conditions are met. The score is capped at $0.95$. It is interpreted as a probability only under the illustrative calibration assumptions stated above. The code calculates this value in `calculate_win_probability`.

The weighted fitness used to guide pheromone learning is:

$$
F(s) = w_{\mathrm{win}}P(s)
+ w_{\mathrm{cost}}\left(1 - \frac{C(s)}{C_{\max}}\right)
+ w_{\mathrm{risk}}\left(1 - \frac{R(s)}{3}\right),
$$

with $w_{\mathrm{win}}=0.5$, $w_{\mathrm{cost}}=0.3$, and $w_{\mathrm{risk}}=0.2$. The value $C_{\max}$ is the greatest additive cost burden attainable from the decision graph: the sum of the largest option cost in each category. The cost-efficiency term is normalized using this finite graph; this does not make the modeled cost burden monetary. The code evaluates the expression in `calculate_fitness`.

For each category-option pair, the construction rule calculates attractiveness and normalized selection probability as follows:

$$
A_{j,k} = \tau_{j,k}^{\alpha}\eta_{j,k}^{\beta},
\qquad
q_{j,k} = \frac{A_{j,k}}{\sum_{\ell \in K_j} A_{j,\ell}}.
$$

Here, $A_{j,k}$ is the attractiveness of option $k$, $\alpha$ controls the influence of pheromone, and $\beta$ controls the influence of heuristic information. The values $q_{j,k}$ form a probability distribution over the options in category $j$. With probability $q_0$, the ant chooses the option with the largest $q_{j,k}$; otherwise, it samples according to that distribution. This calculation and selection occur in `construct_solution`.

After an iteration's strategies are evaluated, pheromone is updated according to:

$$
\tau_{j,k} \leftarrow (1-\rho)\tau_{j,k}
+ \sum_{s \in S: k_j(s)=k} F(s),
$$

where $S$ is the set of strategies constructed in that iteration, $k_j(s)$ is strategy $s$'s selected option in category $j$, and $\rho$ is the evaporation rate. Only positive-fitness strategies deposit pheromone in the implementation. The corresponding code is `update_pheromones`.

The Pareto analysis treats win probability as an objective to maximize and cost and risk as objectives to minimize. For two strategies $s$ and $t$, strategy $s$ dominates $t$ if

$$
P(s) \ge P(t),\qquad C(s) \le C(t),\qquad R(s) \le R(t),
$$

and at least one of these three comparisons is strict. A non-dominated strategy is one for which no other observed strategy satisfies this dominance condition. The functions `is_pareto_optimal` and `update_pareto_frontier` maintain this set of observed trade-offs; they do not enumerate strategies that the stochastic search did not encounter.

The algorithm constructs $A$ strategies per iteration, where $A$ is the number of ants. If there are $|J|$ categories and at most $K$ options per category, construction and pheromone updates require $O(A|J|K)$ work per iteration. The number of possible complete strategies is $\prod_{j\in J}|K_j|$. Pareto-frontier maintenance compares objective vectors pairwise; its worst-case work is quadratic in the number of distinct objective vectors encountered. Retaining only one representative of each tied objective vector prevents repeated observations of the same trade-off from expanding the frontier.

### Approach

1. Represent each decision option and complete strategy with data classes.
2. Build the decision graph and initialize pheromone and heuristic values.
3. Construct strategies probabilistically, calculate their objective values, and update pheromones.
4. Track the best-so-far weighted fitness and maintain the non-dominated strategies observed by the search.
5. Visualize convergence, pheromone evolution, objective trade-offs, strategy components, and solution diversity.
6. Examine a small set of ACO parameter configurations using the same decision graph and algorithm.

### Interpreting the results

A rising best-so-far fitness indicates that the search has encountered a better weighted solution; a flat curve indicates no improvement during those iterations, not proof of optimality. Pheromone histories summarize average trail levels by category, not preferences for individual options. The Pareto plot displays modeled cost burden and win-probability score, with color indicating risk; a point can be preferable on one objective and worse on another. Parameter-sensitivity results describe these particular stochastic runs and do not establish universal parameter settings.

The main example uses 50 ants and 100 iterations. Sensitivity runs use 30 ants and 50 iterations for each listed parameter configuration. Numerical outcomes are produced when the notebook is executed.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class BiddingDecision:
    """One available option in a bidding decision category."""

    decision_type: str
    option_id: int
    value: float
    cost: float
    risk: float
    win_probability_contribution: float


@dataclass
class BiddingStrategy:
    """A complete strategy and its evaluated model objectives."""

    decisions: List[BiddingDecision]
    total_cost: float
    total_risk: float
    win_probability: float
    fitness_score: float


@dataclass
class ACOParameters:
    """Parameters controlling the ACO search."""

    n_ants: int = 50
    n_iterations: int = 100
    alpha: float = 1.0
    beta: float = 2.0
    rho: float = 0.1
    q0: float = 0.8
    tau_0: float = 0.1


@dataclass
class ACOResults:
    """Search histories and solutions returned by the ACO run."""

    best_strategy: BiddingStrategy
    convergence_history: List[float]
    pheromone_evolution: Dict[str, List[float]]
    pareto_frontier: List[BiddingStrategy]
    execution_time: float
    parameter_sensitivity: Dict

In [ ]:
class ConcessionBiddingACO:
    """ACO search over the concession-bidding decision graph."""

    def __init__(self, parameters: ACOParameters):
        self.params = parameters
        self.decision_graph = {}
        self.pheromone_trails = {}
        self.heuristic_info = {}
        self.best_solutions = []
        self.cost_scale = 1.0

    def initialize_decision_graph(self):
        """Create the option graph and initialize option-level search data."""
        self.decision_graph = {
            "bid_amount": [
                BiddingDecision("bid_amount", 0, 75.0, 75.0, 0.3, 0.65),
                BiddingDecision("bid_amount", 1, 87.5, 87.5, 0.5, 0.78),
                BiddingDecision("bid_amount", 2, 100.0, 100.0, 0.7, 0.85),
            ],
            "performance_commitment": [
                BiddingDecision("performance_commitment", 0, 1.4, 1.4, 0.2, 0.70),
                BiddingDecision("performance_commitment", 1, 1.6, 1.6, 0.4, 0.82),
                BiddingDecision("performance_commitment", 2, 1.8, 1.8, 0.6, 0.88),
            ],
            "technology_specification": [
                BiddingDecision("technology_specification", 0, 45.0, 45.0, 0.2, 0.75),
                BiddingDecision("technology_specification", 1, 65.0, 65.0, 0.4, 0.85),
            ],
            "employment_commitment": [
                BiddingDecision("employment_commitment", 0, 280, 280, 0.1, 0.70),
                BiddingDecision("employment_commitment", 1, 320, 320, 0.2, 0.80),
                BiddingDecision("employment_commitment", 2, 360, 360, 0.3, 0.85),
            ],
            "timeline_commitment": [
                BiddingDecision("timeline_commitment", 0, 24, 24, 0.3, 0.75),
                BiddingDecision("timeline_commitment", 1, 22, 22, 0.4, 0.82),
                BiddingDecision("timeline_commitment", 2, 20, 20, 0.5, 0.88),
            ],
        }

        self.cost_scale = sum(
            max(option.cost for option in options)
            for options in self.decision_graph.values()
        )
        self.pheromone_trails = {
            decision_type: {
                option.option_id: self.params.tau_0 for option in options
            }
            for decision_type, options in self.decision_graph.items()
        }
        self.calculate_heuristic_info()

    def calculate_heuristic_info(self):
        """Compute option-specific guidance from the model inputs."""
        self.heuristic_info = {}

        for decision_type, options in self.decision_graph.items():
            self.heuristic_info[decision_type] = {}
            for option in options:
                if decision_type in {"bid_amount", "technology_specification"}:
                    heuristic = option.win_probability_contribution / (
                        1 + option.cost / 100
                    )
                elif decision_type in {
                    "performance_commitment",
                    "employment_commitment",
                }:
                    heuristic = option.win_probability_contribution
                else:
                    heuristic = option.win_probability_contribution / (
                        1 + option.risk
                    )

                self.heuristic_info[decision_type][option.option_id] = heuristic

    def construct_solution(self, ant_id: int) -> BiddingStrategy:
        """Construct and evaluate one complete strategy."""
        decisions = []
        total_cost = 0.0
        total_risk = 0.0

        for decision_type, options in self.decision_graph.items():
            probabilities = []
            for option in options:
                pheromone = self.pheromone_trails[decision_type][option.option_id]
                heuristic = self.heuristic_info[decision_type][option.option_id]
                attractiveness = (
                    pheromone ** self.params.alpha
                ) * (heuristic ** self.params.beta)
                probabilities.append(attractiveness)

            probability_sum = sum(probabilities)
            if probability_sum > 0:
                probabilities = [value / probability_sum for value in probabilities]
            else:
                probabilities = [1.0 / len(options)] * len(options)

            if random.random() < self.params.q0:
                selected_index = int(np.argmax(probabilities))
            else:
                selected_index = int(
                    np.random.choice(len(options), p=probabilities)
                )

            selected_option = options[selected_index]
            decisions.append(selected_option)
            total_cost += selected_option.cost
            total_risk += selected_option.risk

        win_probability = self.calculate_win_probability(decisions)
        fitness_score = self.calculate_fitness(
            decisions, total_cost, total_risk, win_probability
        )

        return BiddingStrategy(
            decisions=decisions,
            total_cost=total_cost,
            total_risk=total_risk,
            win_probability=win_probability,
            fitness_score=fitness_score,
        )

    def calculate_win_probability(
        self, decisions: List[BiddingDecision]
    ) -> float:
        """Combine option contributions and the specified interaction bonuses."""
        base_probability = sum(
            decision.win_probability_contribution * 0.2
            for decision in decisions
        )

        bid_amount = next(
            (d for d in decisions if d.decision_type == "bid_amount"), None
        )
        performance = next(
            (d for d in decisions
             if d.decision_type == "performance_commitment"),
            None,
        )
        technology = next(
            (d for d in decisions
             if d.decision_type == "technology_specification"),
            None,
        )

        synergy_bonus = 0.0
        if bid_amount and performance:
            if bid_amount.option_id >= 1 and performance.option_id >= 1:
                synergy_bonus += 0.1

        if technology and performance:
            if technology.option_id == 1 and performance.option_id >= 1:
                synergy_bonus += 0.08

        return min(0.95, base_probability + synergy_bonus)

    def calculate_fitness(
        self,
        decisions: List[BiddingDecision],
        total_cost: float,
        total_risk: float,
        win_probability: float,
    ) -> float:
        """Evaluate the weighted win, cost-efficiency, and risk objectives."""
        w_win = 0.5
        w_cost = 0.3
        w_risk = 0.2

        normalized_cost = 1.0 - total_cost / self.cost_scale
        normalized_risk = 1.0 - total_risk / 3.0

        return (
            w_win * win_probability
            + w_cost * normalized_cost
            + w_risk * normalized_risk
        )

    def update_pheromones(self, solutions: List[BiddingStrategy]):
        """Evaporate existing trails and reinforce used options."""
        for decision_type, trails in self.pheromone_trails.items():
            for option_id in trails:
                trails[option_id] *= 1 - self.params.rho

        for solution in solutions:
            if solution.fitness_score > 0:
                for decision in solution.decisions:
                    self.pheromone_trails[decision.decision_type][
                        decision.option_id
                    ] += solution.fitness_score

    def optimize(self) -> ACOResults:
        """Run the ACO iterations and return their search records."""
        start_time = time.time()
        self.best_solutions = []
        self.initialize_decision_graph()

        convergence_history = []
        pheromone_evolution = {
            decision_type: [] for decision_type in self.decision_graph
        }
        pareto_frontier = []

        for iteration in range(self.params.n_iterations):
            solutions = [
                self.construct_solution(ant)
                for ant in range(self.params.n_ants)
            ]
            iteration_best = max(solutions, key=lambda s: s.fitness_score)

            if (
                not self.best_solutions
                or iteration_best.fitness_score
                > self.best_solutions[-1].fitness_score
            ):
                self.best_solutions.append(iteration_best)

            best_fitness = self.best_solutions[-1].fitness_score
            convergence_history.append(best_fitness)

            for decision_type in self.decision_graph:
                pheromone_evolution[decision_type].append(
                    float(np.mean(list(
                        self.pheromone_trails[decision_type].values()
                    )))
                )

            for solution in solutions:
                if is_pareto_optimal(solution, pareto_frontier):
                    pareto_frontier.append(solution)
                    pareto_frontier = update_pareto_frontier(pareto_frontier)

            self.update_pheromones(solutions)

            if iteration % 20 == 0:
                print(
                    f"Iteration {iteration}: "
                    f"Best fitness = {best_fitness:.4f}"
                )

        return ACOResults(
            best_strategy=self.best_solutions[-1] if self.best_solutions else None,
            convergence_history=convergence_history,
            pheromone_evolution=pheromone_evolution,
            pareto_frontier=pareto_frontier,
            execution_time=time.time() - start_time,
            parameter_sensitivity={},
        )

In [ ]:
def _same_objectives(
    first: BiddingStrategy,
    second: BiddingStrategy,
) -> bool:
    """Compare objective vectors using tolerances for floating-point values."""
    return (
        math.isclose(
            first.win_probability,
            second.win_probability,
            rel_tol=1e-9,
            abs_tol=1e-12,
        )
        and math.isclose(
            first.total_cost,
            second.total_cost,
            rel_tol=1e-9,
            abs_tol=1e-12,
        )
        and math.isclose(
            first.total_risk,
            second.total_risk,
            rel_tol=1e-9,
            abs_tol=1e-12,
        )
    )


def _dominates(
    first: BiddingStrategy,
    second: BiddingStrategy,
) -> bool:
    """Test Pareto dominance with win maximized and cost and risk minimized."""
    no_worse = (
        (first.win_probability > second.win_probability
         or math.isclose(
             first.win_probability,
             second.win_probability,
             rel_tol=1e-9,
             abs_tol=1e-12,
         ))
        and (first.total_cost < second.total_cost
             or math.isclose(
                 first.total_cost,
                 second.total_cost,
                 rel_tol=1e-9,
                 abs_tol=1e-12,
             ))
        and (first.total_risk < second.total_risk
             or math.isclose(
                 first.total_risk,
                 second.total_risk,
                 rel_tol=1e-9,
                 abs_tol=1e-12,
             ))
    )
    strictly_better = (
        (first.win_probability > second.win_probability
         and not math.isclose(
             first.win_probability,
             second.win_probability,
             rel_tol=1e-9,
             abs_tol=1e-12,
         ))
        or (first.total_cost < second.total_cost
            and not math.isclose(
                first.total_cost,
                second.total_cost,
                rel_tol=1e-9,
                abs_tol=1e-12,
            ))
        or (first.total_risk < second.total_risk
            and not math.isclose(
                first.total_risk,
                second.total_risk,
                rel_tol=1e-9,
                abs_tol=1e-12,
            ))
    )
    return no_worse and strictly_better


def is_pareto_optimal(
    strategy: BiddingStrategy,
    frontier: List[BiddingStrategy],
) -> bool:
    """Return whether the strategy adds a distinct non-dominated objective vector."""
    for existing in frontier:
        if _same_objectives(existing, strategy) or _dominates(existing, strategy):
            return False
    return True


def update_pareto_frontier(
    frontier: List[BiddingStrategy],
) -> List[BiddingStrategy]:
    """Keep one representative of each non-dominated objective vector."""
    unique_strategies = []
    for strategy in frontier:
        if not any(
            _same_objectives(strategy, existing)
            for existing in unique_strategies
        ):
            unique_strategies.append(strategy)

    non_dominated = []
    for index, strategy in enumerate(unique_strategies):
        if not any(
            other_index != index and _dominates(other, strategy)
            for other_index, other in enumerate(unique_strategies)
        ):
            non_dominated.append(strategy)

    return non_dominated

In [ ]:
def run_aco_optimization():
    """Configure and run the main terminal-concession ACO example."""
    print("Ant Colony Optimization for Terminal Concession Bidding")

    params = ACOParameters(
        n_ants=50,
        n_iterations=100,
        alpha=1.0,
        beta=2.0,
        rho=0.1,
        q0=0.8,
        tau_0=0.1,
    )

    print("\nACO parameters:")
    print(f"- Number of ants: {params.n_ants}")
    print(f"- Number of iterations: {params.n_iterations}")
    print(f"- Alpha (pheromone influence): {params.alpha}")
    print(f"- Beta (heuristic influence): {params.beta}")
    print(f"- Rho (evaporation rate): {params.rho}")
    print(f"- Q0 (exploitation parameter): {params.q0}")

    aco = ConcessionBiddingACO(params)
    print("\nStarting ACO search...")
    results = aco.optimize()
    return results, aco


results, aco = run_aco_optimization()

In [ ]:
def analyze_optimal_solution(
    results: ACOResults,
    aco: ConcessionBiddingACO,
):
    """Display the best weighted-fitness strategy and its objectives."""
    strategy = results.best_strategy
    if strategy is None:
        print("No strategy was constructed.")
        return None

    print("\nBest weighted-fitness strategy found")
    print(f"- Fitness score: {strategy.fitness_score:.4f}")
    print(f"- Win-probability score: {strategy.win_probability:.2%}")
    print(f"- Total modeled cost burden: {strategy.total_cost:.1f} (model units)")
    print(f"- Total risk index: {strategy.total_risk:.2f}")

    print("\nStrategy components:")
    for decision in strategy.decisions:
        if decision.decision_type == "bid_amount":
            print(f"- Upfront payment option: {decision.value:.1f} million euros")
        elif decision.decision_type == "performance_commitment":
            print(
                "- Performance commitment: "
                f"{decision.value:.1f} million TEU annually"
            )
        elif decision.decision_type == "technology_specification":
            technology = (
                "Basic automation"
                if decision.option_id == 0
                else "Advanced automation systems"
            )
            print(f"- Technology: {technology}; value {decision.value:.1f}")
        elif decision.decision_type == "employment_commitment":
            print(f"- Employment commitment: {int(decision.value)} jobs")
        elif decision.decision_type == "timeline_commitment":
            print(
                "- Timeline commitment: "
                f"{int(decision.value)} months to operational readiness"
            )

    cost_efficiency = 1.0 - strategy.total_cost / aco.cost_scale
    risk_management = 1.0 - strategy.total_risk / 3.0
    print("\nWeighted-fitness components:")
    print(f"- Win probability (weight 0.5): {strategy.win_probability:.3f}")
    print(f"- Cost efficiency (weight 0.3): {cost_efficiency:.3f}")
    print(f"- Risk management (weight 0.2): {risk_management:.3f}")
    print("\nSearch characteristics:")
    print(f"- Execution time: {results.execution_time:.2f} seconds")
    print(f"- Iterations recorded: {len(results.convergence_history)}")
    print(f"- Non-dominated strategies retained: {len(results.pareto_frontier)}")
    return strategy


optimal_strategy = analyze_optimal_solution(results, aco)

In [ ]:
def visualize_aco_results(
    results: ACOResults,
    aco: ConcessionBiddingACO,
):
    """Plot search progress, objective trade-offs, and strategy summaries."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(
        "ACO: Terminal Concession Bidding",
        fontsize=16,
        fontweight="bold",
    )

    iterations = range(len(results.convergence_history))
    ax = axes[0, 0]
    ax.plot(iterations, results.convergence_history, color="blue", linewidth=2)
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Best-so-far fitness score")
    ax.set_title("Best-Fitness Convergence")
    ax.grid(True, alpha=0.3)
    if len(results.convergence_history) > 10:
        early_average = float(np.mean(results.convergence_history[:10]))
        late_average = float(np.mean(results.convergence_history[-10:]))
        if not np.isclose(early_average, 0.0):
            improvement = (late_average - early_average) / abs(early_average) * 100
            ax.text(
                0.04,
                0.95,
                f"Early-to-late average change: {improvement:.1f}%",
                transform=ax.transAxes,
                va="top",
            )

    ax = axes[0, 1]
    for decision_type, history in results.pheromone_evolution.items():
        ax.plot(
            iterations,
            history,
            label=decision_type.replace("_", " ").title(),
        )
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Average pheromone level")
    ax.set_title("Average Pheromone by Decision Category")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    ax = axes[0, 2]
    if results.pareto_frontier:
        costs = [s.total_cost for s in results.pareto_frontier]
        probabilities = [s.win_probability for s in results.pareto_frontier]
        risks = [s.total_risk for s in results.pareto_frontier]
        points = ax.scatter(
            costs,
            probabilities,
            c=risks,
            s=70,
            cmap="viridis",
            alpha=0.8,
        )
        fig.colorbar(points, ax=ax, label="Total risk index")
        if results.best_strategy:
            ax.scatter(
                results.best_strategy.total_cost,
                results.best_strategy.win_probability,
                color="red",
                marker="*",
                s=180,
                edgecolors="black",
                label="Best weighted-fitness strategy",
            )
            ax.legend(fontsize=8)
    ax.set_xlabel("Total modeled cost burden (model units)")
    ax.set_ylabel("Win-probability score")
    ax.set_title("Non-Dominated Objective Trade-Offs")
    ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    if results.best_strategy:
        labels = [
            decision.decision_type.replace("_", " ").title()
            for decision in results.best_strategy.decisions
        ]
        values = [decision.value for decision in results.best_strategy.decisions]
        bars = ax.bar(labels, values, alpha=0.85)
        for bar, value in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                f"{value:g}",
                ha="center",
                va="bottom",
                fontsize=8,
            )
    ax.set_xlabel("Decision category")
    ax.set_ylabel("Decision value (native units)")
    ax.set_title("Best Strategy Components")
    ax.tick_params(axis="x", rotation=35)
    ax.grid(True, axis="y", alpha=0.3)

    ax = axes[1, 1]
    objectives = ["Win probability", "Cost efficiency", "Risk management"]
    angles = np.linspace(0, 2 * np.pi, len(objectives), endpoint=False).tolist()
    angles += angles[:1]
    if results.best_strategy:
        strategy = results.best_strategy
        values = [
            strategy.win_probability,
            1.0 - strategy.total_cost / aco.cost_scale,
            1.0 - strategy.total_risk / 3.0,
        ]
        values += values[:1]
        ax.plot(angles, values, "o-", color="red", label="Best weighted fitness")
        ax.fill(angles, values, color="red", alpha=0.15)
    for index, strategy in enumerate(results.pareto_frontier[:3]):
        values = [
            strategy.win_probability,
            1.0 - strategy.total_cost / aco.cost_scale,
            1.0 - strategy.total_risk / 3.0,
        ]
        values += values[:1]
        ax.plot(angles, values, "--", label=f"Pareto strategy {index + 1}")
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(objectives, fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title("Normalized Objective Scores")
    ax.legend(fontsize=7, loc="upper right")
    ax.grid(True)

    ax = axes[1, 2]
    if results.pareto_frontier:
        ranges = {
            "Cost burden": max(s.total_cost for s in results.pareto_frontier)
            - min(s.total_cost for s in results.pareto_frontier),
            "Win probability": max(
                s.win_probability for s in results.pareto_frontier
            ) - min(s.win_probability for s in results.pareto_frontier),
            "Risk index": max(s.total_risk for s in results.pareto_frontier)
            - min(s.total_risk for s in results.pareto_frontier),
        }
        bars = ax.bar(list(ranges), list(ranges.values()), alpha=0.85)
        for bar, value in zip(bars, ranges.values()):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                f"{value:.3f}",
                ha="center",
                va="bottom",
                fontsize=8,
            )
    ax.set_ylabel("Range across retained non-dominated strategies")
    ax.set_title("Observed Solution Diversity")
    ax.tick_params(axis="x", rotation=20)
    ax.grid(True, axis="y", alpha=0.3)

    fig.tight_layout()
    plt.show()
    return fig


fig = visualize_aco_results(results, aco)

In [ ]:
def parameter_sensitivity_analysis():
    """Compare the specified ACO parameter configurations."""
    print("\nParameter Sensitivity Analysis")

    parameter_sets = [
        {"alpha": 0.5, "beta": 2.0, "rho": 0.1, "name": "Low Pheromone"},
        {"alpha": 1.0, "beta": 2.0, "rho": 0.1, "name": "Balanced"},
        {"alpha": 2.0, "beta": 1.0, "rho": 0.1, "name": "High Pheromone"},
        {"alpha": 1.0, "beta": 1.0, "rho": 0.05, "name": "Low Evaporation"},
        {"alpha": 1.0, "beta": 3.0, "rho": 0.2, "name": "High Heuristic"},
    ]
    sensitivity_results = []

    for parameter_set in parameter_sets:
        print(f"\nTesting: {parameter_set['name']}")
        params = ACOParameters(
            n_ants=30,
            n_iterations=50,
            alpha=parameter_set["alpha"],
            beta=parameter_set["beta"],
            rho=parameter_set["rho"],
            q0=0.8,
            tau_0=0.1,
        )
        aco_test = ConcessionBiddingACO(params)
        test_results = aco_test.optimize()
        strategy = test_results.best_strategy

        sensitivity_results.append(
            {
                "Parameter Set": parameter_set["name"],
                "Alpha": parameter_set["alpha"],
                "Beta": parameter_set["beta"],
                "Rho": parameter_set["rho"],
                "Best Fitness": strategy.fitness_score if strategy else 0.0,
                "Win Probability": strategy.win_probability if strategy else 0.0,
                "Total Cost": strategy.total_cost if strategy else 0.0,
                "Execution Time": test_results.execution_time,
                "Convergence Rate": len(test_results.convergence_history),
            }
        )
        print(f"  Best fitness: {sensitivity_results[-1]['Best Fitness']:.4f}")
        print(f"  Execution time: {test_results.execution_time:.2f} seconds")

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("ACO Parameter Sensitivity Analysis", fontsize=16)
    labels = [result["Parameter Set"] for result in sensitivity_results]
    positions = range(len(sensitivity_results))

    axes[0, 0].bar(
        positions,
        [result["Best Fitness"] for result in sensitivity_results],
        color="skyblue",
    )
    axes[0, 0].set_ylabel("Best fitness score")
    axes[0, 0].set_title("Weighted Fitness")

    axes[0, 1].bar(
        positions,
        [result["Win Probability"] for result in sensitivity_results],
        color="lightcoral",
    )
    axes[0, 1].set_ylabel("Win-probability score")
    axes[0, 1].set_title("Win-Probability Objective")

    axes[1, 0].bar(
        positions,
        [result["Execution Time"] for result in sensitivity_results],
        color="lightgreen",
    )
    axes[1, 0].set_ylabel("Execution time (seconds)")
    axes[1, 0].set_title("Runtime")

    parameter_matrix = np.array(
        [
            [result["Alpha"] for result in sensitivity_results],
            [result["Beta"] for result in sensitivity_results],
            [result["Rho"] for result in sensitivity_results],
        ]
    )
    image = axes[1, 1].imshow(parameter_matrix, cmap="viridis", aspect="auto")
    axes[1, 1].set_yticks(range(3))
    axes[1, 1].set_yticklabels(["Alpha", "Beta", "Rho"])
    axes[1, 1].set_title("Parameter Values")
    fig.colorbar(image, ax=axes[1, 1], label="Parameter value")
    for row in range(parameter_matrix.shape[0]):
        for column in range(parameter_matrix.shape[1]):
            axes[1, 1].text(
                column,
                row,
                f"{parameter_matrix[row, column]:.2f}",
                ha="center",
                va="center",
                color="white",
            )

    for ax in axes.flat[:3]:
        ax.set_xlabel("Parameter configuration")
        ax.set_xticks(list(positions))
        ax.set_xticklabels(labels, rotation=35, ha="right")
        ax.grid(True, axis="y", alpha=0.3)
    axes[1, 1].set_xticks(list(positions))
    axes[1, 1].set_xticklabels(labels, rotation=35, ha="right")

    fig.tight_layout()
    plt.show()
    return sensitivity_results


sensitivity_results = parameter_sensitivity_analysis()

In [ ]:
def generate_aco_summary(
    results: ACOResults,
    sensitivity_results: List[Dict],
):
    """Summarize the search results and the tested parameter configurations."""
    print("\n" + "=" * 72)
    print("TERMINAL CONCESSION BIDDING: ACO SUMMARY")
    print("=" * 72)
    print("\nAlgorithm characteristics:")
    print("- Probabilistic strategy construction by a population of ants")
    print("- Option selection guided by pheromone and heuristic information")
    print("- Pheromone learning through evaporation and fitness-based deposition")
    print("- Weighted fitness and a separate Pareto-dominance analysis")

    if results.best_strategy:
        strategy = results.best_strategy
        print("\nBest weighted-fitness strategy found:")
        print(f"- Fitness score: {strategy.fitness_score:.4f}")
        print(f"- Win-probability score: {strategy.win_probability:.2%}")
        print(
            "- Total modeled cost burden: "
            f"{strategy.total_cost:.1f} (model units)"
        )
        print(f"- Total risk index: {strategy.total_risk:.2f}")
        print(f"- Search execution time: {results.execution_time:.2f} seconds")
        print("\nSelected options:")
        for decision in strategy.decisions:
            print(
                f"- {decision.decision_type.replace('_', ' ').title()}: "
                f"{decision.value:g}"
            )

    print("\nSearch and trade-off records:")
    print(f"- Iterations recorded: {len(results.convergence_history)}")
    print(
        "- Non-dominated strategies retained: "
        f"{len(results.pareto_frontier)}"
    )
    if results.pareto_frontier:
        cost_range = max(s.total_cost for s in results.pareto_frontier) - min(
            s.total_cost for s in results.pareto_frontier
        )
        probability_range = max(
            s.win_probability for s in results.pareto_frontier
        ) - min(s.win_probability for s in results.pareto_frontier)
        risk_range = max(s.total_risk for s in results.pareto_frontier) - min(
            s.total_risk for s in results.pareto_frontier
        )
        print(f"- Cost-burden range: {cost_range:.2f} model units")
        print(f"- Win-probability-score range: {probability_range:.3f}")
        print(f"- Risk-index range: {risk_range:.2f}")

    if sensitivity_results:
        best_parameters = max(
            sensitivity_results, key=lambda item: item["Best Fitness"]
        )
        worst_parameters = min(
            sensitivity_results, key=lambda item: item["Best Fitness"]
        )
        variation = (
            best_parameters["Best Fitness"] - worst_parameters["Best Fitness"]
        )
        print("\nParameter sensitivity for these stochastic runs:")
        print(
            f"- Highest observed fitness: {best_parameters['Parameter Set']} "
            f"({best_parameters['Best Fitness']:.4f})"
        )
        print(
            f"- Lowest observed fitness: {worst_parameters['Parameter Set']} "
            f"({worst_parameters['Best Fitness']:.4f})"
        )
        print(f"- Observed fitness range: {variation:.4f}")
        print(
            "These observations describe the tested runs; stochastic results "
            "do not establish a universally best parameter setting."
        )

    print("\nInterpretation:")
    print(
        "The best weighted-fitness strategy reflects the chosen objective "
        "weights. The Pareto frontier shows observed non-dominated trade-offs, "
        "and neither record proves global optimality. Cost values are model "
        "units rather than monetary totals."
    )
    print("\n" + "=" * 72)


generate_aco_summary(results, sensitivity_results)